# Pandas Data-Wrangling Workout

A hands-on workbook for brushing up pandas through **59 exercises** on a deliberately
messy e-commerce dataset. The emphasis is on pure functions, method chaining with
`.pipe`, explicit dtypes, validated joins, and vectorized group operations.

The workbook covers the full data-wrangling loop:

1. inspect and type data;
2. clean, validate, and quarantine bad rows;
3. engineer features from numbers, strings, and dates;
4. aggregate, join, and reshape;
5. work with time series and MultiIndexes;
6. profile performance and refactor imperative code;
7. finish with a functional capstone and a challenge set.

## How to use this workbook

- Run the **Setup** cell first.
- Work through the sections in order.
- Each exercise has a prompt, a starter cell, a self-check cell, and a collapsible
  solution.
- Run the self-check after writing your function. A successful check prints a
  `passed` message.
- Try to solve the problem before opening the solution. The checks test behavior,
  so alternative correct implementations are welcome.
- The dataset is committed under `datasets/`; no downloads are required.

## Functional ground rules

- Treat DataFrames as immutable values at the boundary of each function.
- Return new objects instead of using `inplace=True`.
- Avoid chained assignment and `SettingWithCopyWarning`.
- Prefer `assign`, `query`, `loc`, named aggregation, `transform`, and `.pipe`.
- Prefer vectorized expressions over `iterrows` and row-wise `apply`.
- Be explicit about merge cardinality with `validate=`.
- Validate results with assertions and invariants, not only by eyeballing output.
- Keep helper functions small, named, and independently testable.

The setup enables pandas copy-on-write, which makes the purity rules easier to follow
and matches the direction of pandas 3.0.

## Dataset

The workbook uses five related CSV files.

| Table | Grain | Rows | Key columns |
| --- | --- | --- | --- |
| `customers.csv` | one row per customer | 215 | `customer_id`, `email`, `signup_date` |
| `products.csv` | one row per product | 30 | `product_id`, `category`, `unit_cost` |
| `product_prices.csv` | one price change per product/date | 62 | `product_id`, `effective_date`, `list_price` |
| `orders.csv` | order lines, including duplicate updates | 1,770 | `order_id`, `customer_id`, `product_id`, `order_date` |
| `returns.csv` | return events, including duplicates | 170 | `return_id`, `order_id`, `return_date` |

### Dirty-data inventory

The data contains deliberate problems so every cleaning skill has something real to
work on:

- country, segment, category, status, channel, reason, and active-flag variants;
- mixed date formats, missing dates, and impossible dates;
- currency-formatted prices (`$1,299.00`, `USD 50.21`) and missing prices;
- missing and orphan customer/product foreign keys;
- exact duplicate rows and conflicting duplicate `order_id` updates;
- missing quantities, negative quantities, zeros, and large outliers;
- missing discounts, negative discounts, and discounts above 50%;
- customers with no orders, orders with no matching product, and orphan returns.

Use the first few rows of each table to orient yourself, then rely on the audit
exercises to quantify the problems.

## Setup

Run this cell before anything else.

In [ ]:
from __future__ import annotations

import tempfile
import time
from pathlib import Path

import numpy as np
import pandas as pd

pd.options.mode.copy_on_write = True
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.max_rows", 100)

DATA_DIR = Path("datasets")
RANDOM_STATE = 42

CUSTOMERS = pd.read_csv(DATA_DIR / "customers.csv", dtype={"customer_id": "string"})
PRODUCTS = pd.read_csv(DATA_DIR / "products.csv", dtype={"product_id": "string"})
PRICES = pd.read_csv(DATA_DIR / "product_prices.csv", dtype={"product_id": "string"})
ORDERS = pd.read_csv(
    DATA_DIR / "orders.csv",
    dtype={"order_id": "string", "customer_id": "string", "product_id": "string"},
)
RETURNS = pd.read_csv(
    DATA_DIR / "returns.csv",
    dtype={"return_id": "string", "order_id": "string"},
)

RAW = {
    "customers": CUSTOMERS,
    "products": PRODUCTS,
    "prices": PRICES,
    "orders": ORDERS,
    "returns": RETURNS,
}

## 1. Foundations and inspection

Before changing data, learn what is in it. This section focuses on the shape of a
table, column types, missingness, label-based versus position-based selection, and
the difference between a view and an independent copy.

The habit to build here is simple: inspect first, transform second, and never update
the caller's DataFrame by accident.

### 1.1 Build a column audit

Write `inspect_table(df)` returning a DataFrame indexed by column name with these
columns: `dtype`, `non_null`, `missing_pct`, `n_unique`, and `sample` (the first
row's value). `missing_pct` should be rounded to two decimals.

Run it on `ORDERS`.

**Hint:** `df.dtypes`, `df.notna().sum()`, `df.isna().mean()`, and `df.nunique()` all return Series indexed by column name, so they align into one DataFrame.

In [ ]:
def inspect_table(df):
    ...

**Self-check**

In [ ]:
out = inspect_table(ORDERS)
assert isinstance(out, pd.DataFrame)
assert list(out.columns) == ["dtype", "non_null", "missing_pct", "n_unique", "sample"]
assert out.index.equals(ORDERS.columns)
assert out.shape == (ORDERS.shape[1], 5)
assert (out["non_null"] + ORDERS.isna().sum()).eq(len(ORDERS)).all()
assert out.loc["quantity", "dtype"] == str(ORDERS["quantity"].dtype)
print("1.1 passed")

<details>
<summary>Show solution</summary>

```python
def inspect_table(df):
    sample = df.iloc[0] if len(df) else pd.Series(index=df.columns, dtype=object)
    return pd.DataFrame(
        {
            "dtype": df.dtypes.astype(str),
            "non_null": df.notna().sum(),
            "missing_pct": df.isna().mean().mul(100).round(2),
            "n_unique": df.nunique(dropna=True),
            "sample": sample,
        }
    )
```

</details>

### 1.2 Select by label, not by accident

Write `slice_orders(df)` that returns rows with index labels 10 through 19
(inclusive) and only the columns `order_id`, `customer_id`, and `quantity`.

The returned frame should be independent of the input frame.

**Hint:** `.loc` uses inclusive labels; `.iloc` uses exclusive positions.

In [ ]:
def slice_orders(df):
    ...

**Self-check**

In [ ]:
out = slice_orders(ORDERS)
assert list(out.columns) == ["order_id", "customer_id", "quantity"]
assert out.index.tolist() == list(range(10, 20))
pd.testing.assert_frame_equal(
    out,
    ORDERS.iloc[10:20][["order_id", "customer_id", "quantity"]],
)
print("1.2 passed")

<details>
<summary>Show solution</summary>

```python
def slice_orders(df):
    return df.loc[10:19, ["order_id", "customer_id", "quantity"]].copy()
```

</details>

### 1.3 Filter with several conditions

Write `filter_orders(df)` that returns completed orders placed in 2024 through the
`web` or `mobile` channel with a quantity of at least 2.

The raw status and channel labels are inconsistent, so normalize them first:
- `complete` becomes `completed`, and `canceled` becomes `cancelled`;
- `online` becomes `web`, `app` becomes `mobile`, and `retail` becomes `store`.

Parse `order_date` with `format="mixed"` and `errors="coerce"`.

**Hint:** Combine masks with `&`, wrap each condition in parentheses, and filter with `.loc[mask]`.

In [ ]:
def filter_orders(df):
    ...

**Self-check**

In [ ]:
out = filter_orders(ORDERS)
assert len(out) == 197
assert list(out.columns) == list(ORDERS.columns)
status = out["status"].astype("string").str.strip().str.lower().replace({"complete": "completed", "canceled": "cancelled"})
channel = out["channel"].astype("string").str.strip().str.lower().replace({"online": "web", "app": "mobile", "retail": "store"})
assert status.eq("completed").all()
assert channel.isin(["web", "mobile"]).all()
assert out["quantity"].ge(2).all()
assert pd.to_datetime(out["order_date"], format="mixed", errors="coerce").dt.year.eq(2024).all()
print("1.3 passed")

<details>
<summary>Show solution</summary>

```python
def filter_orders(df):
    dates = pd.to_datetime(df["order_date"], format="mixed", errors="coerce")
    status = (
        df["status"].astype("string").str.strip().str.lower()
        .replace({"complete": "completed", "canceled": "cancelled"})
    )
    channel = (
        df["channel"].astype("string").str.strip().str.lower()
        .replace({"online": "web", "app": "mobile", "retail": "store"})
    )
    mask = (
        status.eq("completed")
        & channel.isin(["web", "mobile"])
        & df["quantity"].ge(2)
        & dates.dt.year.eq(2024)
    )
    return df.loc[mask].copy()
```

</details>

### 1.4 Return a new frame without mutating the input

Write `normalize_status_copy(df)` that returns a copy with a canonical `status`
column and leaves the input frame untouched.

Canonical values are `completed`, `cancelled`, `pending`, and `returned`.

**Hint:** `df.assign(...)` returns a new DataFrame. Avoid `inplace=True` and chained assignment.

In [ ]:
def normalize_status_copy(df):
    ...

**Self-check**

In [ ]:
before = ORDERS["status"].copy()
out = normalize_status_copy(ORDERS)
assert out is not ORDERS
assert ORDERS["status"].equals(before)
assert set(out["status"].dropna().unique()) == {"completed", "cancelled", "pending", "returned"}
assert out.index.equals(ORDERS.index)
print("1.4 passed")

<details>
<summary>Show solution</summary>

```python
def normalize_status_copy(df):
    status = (
        df["status"].astype("string").str.strip().str.lower()
        .replace({"complete": "completed", "canceled": "cancelled"})
    )
    return df.assign(status=status)
```

</details>

## 2. Loading and typing

Most real-world problems start with a messy CSV. This section covers explicit
`read_csv` options, currency parsing, mixed date formats, memory-aware dtypes, and
round-tripping to JSON and Parquet.

When a loader is explicit about types, every later transformation becomes simpler
and easier to test.

### 2.1 Write a typed orders loader

Write `load_orders(path)` that reads the orders CSV with:
- `order_id`, `customer_id`, and `product_id` as the pandas `"string"` dtype;
- `"N/A"`, `"NA"`, `"?"`, and empty strings treated as missing.

Keep all columns. Do not parse dates in this exercise.

**Hint:** `na_values` is applied on top of pandas' default missing-value list.

In [ ]:
def load_orders(path):
    ...

**Self-check**

In [ ]:
out = load_orders(DATA_DIR / "orders.csv")
assert len(out) == 1770
assert str(out["order_id"].dtype) == "string"
assert str(out["customer_id"].dtype) == "string"
assert str(out["product_id"].dtype) == "string"
assert out["order_date"].dtype == object
assert "updated_at" in out.columns
assert list(out.columns) == list(ORDERS.columns)
print("2.1 passed")

<details>
<summary>Show solution</summary>

```python
def load_orders(path):
    return pd.read_csv(
        path,
        dtype={
            "order_id": "string",
            "customer_id": "string",
            "product_id": "string",
        },
        na_values=["", "N/A", "NA", "?"],
    )
```

</details>

### 2.2 Parse currency strings

Write `parse_currency(series)` returning a `Float64` Series. It should handle
dollar signs, thousands separators, a leading `USD`, surrounding whitespace, empty
strings, and `"N/A"`. Invalid strings become missing.

Apply it to `ORDERS["unit_price"]`.

**Hint:** `Series.str.replace(..., regex=False)` is safer when replacing literal `$` and `,`.

In [ ]:
def parse_currency(series):
    ...

**Self-check**

In [ ]:
sample = pd.Series(["$1,299.00", "19.99", "USD 5.50", "", "N/A"], index=list("abcde"))
parsed = parse_currency(sample)
assert parsed.loc["a"] == 1299.0
assert parsed.loc["b"] == 19.99
assert parsed.loc["c"] == 5.50
assert parsed.loc[["d", "e"]].isna().all()
assert str(parsed.dtype) == "Float64"
actual = parse_currency(ORDERS["unit_price"])
assert actual.notna().sum() == 1748
assert round(float(actual.sum()), 2) == 490834.34
print("2.2 passed")

<details>
<summary>Show solution</summary>

```python
def parse_currency(series):
    cleaned = (
        series.astype("string")
        .str.replace("$", "", regex=False)
        .str.replace("USD", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    return pd.to_numeric(cleaned, errors="coerce").astype("Float64")
```

</details>

### 2.3 Parse mixed date formats

Write `parse_mixed_dates(series)` that parses ISO dates, `MM/DD/YYYY`, `DD-Mon-YYYY`,
and `YYYY/MM/DD` into datetime values. Invalid and missing values become `NaT`.

Apply it to `ORDERS["order_date"]`.

**Hint:** Pandas 2.0+ supports `format="mixed"` for per-value format inference.

In [ ]:
def parse_mixed_dates(series):
    ...

**Self-check**

In [ ]:
sample = pd.Series(["2023-04-15", "04/16/2023", "17-Apr-2023", "2023/04/18", "2023-02-30", ""])
parsed = parse_mixed_dates(sample)
assert parsed.iloc[:4].notna().all()
assert parsed.iloc[4:].isna().all()
actual = parse_mixed_dates(ORDERS["order_date"])
assert actual.isna().sum() == 55
assert str(actual.dtype).startswith("datetime64")
print("2.3 passed")

<details>
<summary>Show solution</summary>

```python
def parse_mixed_dates(series):
    return pd.to_datetime(series, format="mixed", errors="coerce")
```

</details>

### 2.4 Make dtypes cheaper and clearer

Write `optimize_dtypes(df)` returning a copy where:
- all three ID columns use `"string"`;
- `quantity` uses `Int32`;
- `discount` uses `Float32`;
- `status` and `channel` use `category`.

The result must use less memory than the input and preserve the original input.

**Hint:** Check memory with `df.memory_usage(deep=True).sum()`.

In [ ]:
def optimize_dtypes(df):
    ...

**Self-check**

In [ ]:
before_memory = ORDERS.memory_usage(deep=True).sum()
before_quantity = ORDERS["quantity"].dtype
out = optimize_dtypes(ORDERS)
assert str(out["quantity"].dtype) == "Int32"
assert str(out["discount"].dtype) == "Float32"
assert str(out["order_id"].dtype) == "string"
assert isinstance(out["status"].dtype, pd.CategoricalDtype)
assert out.memory_usage(deep=True).sum() < before_memory
assert ORDERS["quantity"].dtype == before_quantity
np.testing.assert_allclose(out["discount"].astype("float64"), ORDERS["discount"], equal_nan=True)
pd.testing.assert_series_equal(out["quantity"].astype("Float64"), ORDERS["quantity"].astype("Float64"))
print("2.4 passed")

<details>
<summary>Show solution</summary>

```python
def optimize_dtypes(df):
    return df.copy().astype(
        {
            "order_id": "string",
            "customer_id": "string",
            "product_id": "string",
            "quantity": "Int32",
            "discount": "Float32",
            "status": "category",
            "channel": "category",
        }
    )
```

</details>

### 2.5 Round-trip JSON and Parquet

Write `roundtrip_io(df)` that writes the frame to a temporary JSON file
(`orient="records"`) and a temporary Parquet file, reads both back, and returns
`(from_json, from_parquet)`.

The temporary directory must disappear before the function returns.

**Hint:** Parquet preserves schema better than JSON; JSON is convenient but often loses dtypes.

In [ ]:
def roundtrip_io(df):
    ...

**Self-check**

In [ ]:
from_json, from_parquet = roundtrip_io(ORDERS)
assert list(from_json.columns) == list(ORDERS.columns)
assert list(from_parquet.columns) == list(ORDERS.columns)
assert len(from_json) == len(ORDERS)
pd.testing.assert_frame_equal(from_parquet, ORDERS, check_dtype=False)
assert from_json.loc[0, "order_id"] == ORDERS.loc[0, "order_id"]
print("2.5 passed")

<details>
<summary>Show solution</summary>

```python
def roundtrip_io(df):
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        df.to_json(tmp / "orders.json", orient="records", date_format="iso")
        df.to_parquet(tmp / "orders.parquet", index=False)
        from_json = pd.read_json(tmp / "orders.json")
        from_parquet = pd.read_parquet(tmp / "orders.parquet")
    return from_json, from_parquet
```

</details>

## 3. Cleaning and validation

Cleaning is not one operation; it is a sequence of small, testable decisions.
Here you normalize labels, deduplicate rows, impute carefully, parse dates, handle
outliers, and separate valid records from a quarantine table.

Every function in this section returns new objects and leaves the raw inputs intact.

### 3.1 Normalize country labels

Write `clean_country(series)` that maps all variants to:
`United States`, `United Kingdom`, `Canada`, `Germany`, or `France`.

Handle `USA`, `US`, `U.S.`, `UK`, `U.K.`, `Great Britain`, `CA`, `CAN`, `DE`,
`Deutschland`, `FR`, case differences, surrounding spaces, missing values, and
the empty string.

**Hint:** Normalize first, then map the normalized strings through a dictionary.

In [ ]:
def clean_country(series):
    ...

**Self-check**

In [ ]:
sample = pd.Series(["USA", " U.K. ", "Deutschland", "CAN", "", None])
parsed = clean_country(sample)
assert parsed.fillna("missing").tolist() == [
    "United States",
    "United Kingdom",
    "Germany",
    "Canada",
    "missing",
    "missing",
]
out = clean_country(CUSTOMERS["country"])
assert out.isna().sum() == 5
assert out.dropna().value_counts().to_dict() == {
    "Canada": 45,
    "France": 44,
    "Germany": 42,
    "United States": 40,
    "United Kingdom": 39,
}
assert out.index.equals(CUSTOMERS.index)
print("3.1 passed")

<details>
<summary>Show solution</summary>

```python
def clean_country(series):
    mapping = {
        "united states": "United States",
        "usa": "United States",
        "us": "United States",
        "u.s.": "United States",
        "united kingdom": "United Kingdom",
        "uk": "United Kingdom",
        "u.k.": "United Kingdom",
        "great britain": "United Kingdom",
        "canada": "Canada",
        "ca": "Canada",
        "can": "Canada",
        "germany": "Germany",
        "de": "Germany",
        "deutschland": "Germany",
        "france": "France",
        "fr": "France",
    }
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .map(mapping)
        .astype("string")
    )
```

</details>

### 3.2 Deduplicate orders by latest update

Some orders appear more than once. Write `dedupe_orders(df)` that keeps exactly one
row per `order_id`, choosing the row with the latest `updated_at` value.

Rows with a missing update timestamp should lose to rows with a valid timestamp.
Parse `updated_at` with `format="mixed"` and `errors="coerce"`.

**Hint:** Sort by timestamp with missing values first, then `drop_duplicates(keep="last")`.

In [ ]:
def dedupe_orders(df):
    ...

**Self-check**

In [ ]:
before = ORDERS.copy(deep=True)
out = dedupe_orders(ORDERS)
assert len(out) == 1700
assert out["order_id"].is_unique
assert out["order_id"].nunique() == 1700
assert list(out.columns) == list(ORDERS.columns)
pd.testing.assert_frame_equal(ORDERS, before)
for order_id in ORDERS.loc[ORDERS["order_id"].duplicated(keep=False), "order_id"].unique():
    assert len(out.loc[out["order_id"] == order_id]) == 1
print("3.2 passed")

<details>
<summary>Show solution</summary>

```python
def dedupe_orders(df):
    updated = pd.to_datetime(df["updated_at"], format="mixed", errors="coerce")
    return (
        df.assign(_updated=updated)
        .sort_values(["order_id", "_updated"], na_position="first")
        .drop_duplicates("order_id", keep="last")
        .drop(columns="_updated")
        .reset_index(drop=True)
    )
```

</details>

### 3.3 Impute with a group median

Write `impute_by_group(df, value_col, group_col)` that fills missing values in
`value_col` with the median of the matching group. Do not alter rows whose group has
no observed values, and do not mutate the input.

Test it on `ORDERS` with `quantity` grouped by `status`.

**Hint:** `groupby(...).transform("median")` returns a Series aligned to the original index.

In [ ]:
def impute_by_group(df, value_col, group_col):
    ...

**Self-check**

In [ ]:
fixture = pd.DataFrame({"g": ["a", "a", "b", "b", "b"], "v": [1.0, np.nan, 10.0, 20.0, np.nan]})
fixture_out = impute_by_group(fixture, "v", "g")
assert fixture_out["v"].tolist() == [1.0, 1.0, 10.0, 20.0, 15.0]
assert fixture["v"].isna().sum() == 2
before_missing = ORDERS["quantity"].isna().sum()
out = impute_by_group(ORDERS, "quantity", "status")
assert ORDERS["quantity"].isna().sum() == before_missing == 12
assert out["quantity"].isna().sum() == 0
print("3.3 passed")

<details>
<summary>Show solution</summary>

```python
def impute_by_group(df, value_col, group_col):
    medians = df.groupby(group_col, observed=True)[value_col].transform("median")
    return df.assign(**{value_col: df[value_col].fillna(medians)})
```

</details>

### 3.4 Parse dates and quarantine bad rows

Write `clean_order_dates(df)` that returns `(clean, bad)`:
- `clean` contains rows with a valid parsed `order_date`, with that column converted
  to datetime and a reset index;
- `bad` contains the rows with missing or invalid dates.

Use `format="mixed"` and `errors="coerce"`.

**Hint:** A boolean mask and its negation partition the rows exactly once.

In [ ]:
def clean_order_dates(df):
    ...

**Self-check**

In [ ]:
clean, bad = clean_order_dates(ORDERS)
assert isinstance(clean, pd.DataFrame) and isinstance(bad, pd.DataFrame)
assert len(clean) + len(bad) == len(ORDERS)
assert len(bad) == 55
assert clean["order_date"].notna().all()
assert bad["order_date"].isna().all()
assert str(clean["order_date"].dtype).startswith("datetime64")
assert clean["order_date"].min() == pd.Timestamp("2022-01-01")
assert clean["order_date"].max() == pd.Timestamp("2024-12-31")
print("3.4 passed")

<details>
<summary>Show solution</summary>

```python
def clean_order_dates(df):
    parsed = pd.to_datetime(df["order_date"], format="mixed", errors="coerce")
    bad_mask = parsed.isna()
    clean = df.loc[~bad_mask].assign(order_date=parsed[~bad_mask]).reset_index(drop=True)
    bad = df.loc[bad_mask].assign(order_date=parsed[bad_mask]).reset_index(drop=True)
    return clean, bad
```

</details>

### 3.5 Clean the products table

Write `clean_products(df)` returning a new frame with:
- `product_name` stripped of surrounding whitespace;
- canonical `category` values: `Electronics`, `Home Office`, `Accessories`,
  `Furniture`;
- `unit_cost` numeric;
- `launch_date` parsed with `format="mixed"`;
- `active` as a nullable boolean (`Y`, `yes`, `1` are true; `N`, `no`, `0` are false).

Leave the input untouched.

**Hint:** Use a mapping dictionary for labels, then cast to nullable boolean.

In [ ]:
def clean_products(df):
    ...

**Self-check**

In [ ]:
before = PRODUCTS.copy(deep=True)
out = clean_products(PRODUCTS)
assert out["product_name"].str.strip().eq(out["product_name"]).all()
assert out["category"].value_counts(dropna=False).to_dict() == {
    "Electronics": 8,
    "Home Office": 8,
    "Accessories": 8,
    "Furniture": 6,
}
assert str(out["active"].dtype) == "boolean"
assert out["active"].value_counts(dropna=False).to_dict() == {True: 15, False: 12, pd.NA: 3}
assert str(out["launch_date"].dtype).startswith("datetime64")
assert out["unit_cost"].isna().sum() == 4
pd.testing.assert_frame_equal(PRODUCTS, before)
print("3.5 passed")

<details>
<summary>Show solution</summary>

```python
def clean_products(df):
    category_map = {
        "electronics": "Electronics",
        "electronic": "Electronics",
        "home office": "Home Office",
        "home & office": "Home Office",
        "accessories": "Accessories",
        "accessory": "Accessories",
        "furniture": "Furniture",
    }
    active_map = {
        "y": True,
        "yes": True,
        "1": True,
        "n": False,
        "no": False,
        "0": False,
    }
    active = (
        df["active"].astype("string").str.strip().str.lower()
        .map(active_map).astype("boolean")
    )
    return df.assign(
        product_name=df["product_name"].str.strip(),
        category=(
            df["category"].astype("string").str.strip().str.lower()
            .map(category_map).astype("string")
        ),
        unit_cost=pd.to_numeric(df["unit_cost"], errors="coerce"),
        launch_date=pd.to_datetime(df["launch_date"], format="mixed", errors="coerce"),
        active=active,
    )
```

</details>

### 3.6 Clip outliers with the IQR rule

Write `clip_iqr(df, value_col)` that clips values to the standard IQR fences:
`Q1 - 1.5 * IQR` and `Q3 + 1.5 * IQR`.

Return a new frame, keep missing values missing, and do not mutate the input. Apply
it to `quantity`.

**Hint:** `Series.clip(lower=..., upper=...)` accepts scalar bounds.

In [ ]:
def clip_iqr(df, value_col):
    ...

**Self-check**

In [ ]:
before = ORDERS["quantity"].copy()
out = clip_iqr(ORDERS, "quantity")
q1 = ORDERS["quantity"].quantile(0.25)
q3 = ORDERS["quantity"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
assert out["quantity"].dropna().between(lower, upper).all()
assert out["quantity"].max() == upper
assert out["quantity"].isna().sum() == ORDERS["quantity"].isna().sum()
assert ORDERS["quantity"].equals(before)
print("3.6 passed")

<details>
<summary>Show solution</summary>

```python
def clip_iqr(df, value_col):
    q1 = df[value_col].quantile(0.25)
    q3 = df[value_col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return df.assign(**{value_col: df[value_col].clip(lower=lower, upper=upper)})
```

</details>

### 3.7 Audit foreign keys

Write `validate_foreign_keys(orders, customers, products)` returning one row per
violation with columns `order_id`, `column`, `value`, and `issue`.

Check `customer_id` against customers and `product_id` against products. Use
`missing customer_id` or `unknown customer_id` (and likewise for products) as the
issue text.

**Hint:** `~Series.isin(valid_keys)` also flags missing keys, because missing is not in the valid set.

In [ ]:
def validate_foreign_keys(orders, customers, products):
    ...

**Self-check**

In [ ]:
out = validate_foreign_keys(ORDERS, CUSTOMERS, PRODUCTS)
assert list(out.columns) == ["order_id", "column", "value", "issue"]
assert len(out) == 122
assert out.loc[out["column"] == "customer_id"].shape[0] == 87
assert out.loc[out["column"] == "product_id"].shape[0] == 35
assert out["issue"].value_counts().to_dict() == {
    "missing customer_id": 51,
    "unknown customer_id": 36,
    "missing product_id": 18,
    "unknown product_id": 17,
}
assert out["order_id"].notna().all()
print("3.7 passed")

<details>
<summary>Show solution</summary>

```python
def validate_foreign_keys(orders, customers, products):
    frames = []
    for column, valid_keys in [
        ("customer_id", customers["customer_id"]),
        ("product_id", products["product_id"]),
    ]:
        bad = orders.loc[~orders[column].isin(valid_keys), ["order_id", column]].copy()
        issue = np.where(
            bad[column].isna(),
            f"missing {column}",
            f"unknown {column}",
        )
        frames.append(
            bad.rename(columns={column: "value"})
            .assign(column=column, issue=issue)
        )
    return pd.concat(frames, ignore_index=True)[["order_id", "column", "value", "issue"]]
```

</details>

### 3.8 Split valid orders from quarantine

Write `split_valid_orders(df, customers, products)` returning `(valid, invalid)`.
A row is valid when:
- `order_date` parses;
- `customer_id` exists in customers;
- `product_id` exists in products;
- `quantity` is positive;
- `unit_price` parses;
- `discount` is between 0 and 0.5 inclusive.

In `valid`, replace `order_date` with the parsed datetime and `unit_price` with the
parsed float. Reset both indexes.

**Hint:** Build one boolean mask, then use it and its negation to partition the rows.

In [ ]:
def split_valid_orders(df, customers, products):
    ...

**Self-check**

In [ ]:
valid, invalid = split_valid_orders(ORDERS, CUSTOMERS, PRODUCTS)
assert len(valid) == 1538
assert len(invalid) == 232
assert valid["order_date"].notna().all()
assert str(valid["order_date"].dtype).startswith("datetime64")
assert valid["unit_price"].notna().all()
assert valid["quantity"].gt(0).all()
assert valid["discount"].between(0, 0.5).all()
assert valid["customer_id"].isin(CUSTOMERS["customer_id"]).all()
assert valid["product_id"].isin(PRODUCTS["product_id"]).all()
assert len(valid) + len(invalid) == len(ORDERS)
print("3.8 passed")

<details>
<summary>Show solution</summary>

```python
def split_valid_orders(df, customers, products):
    dates = pd.to_datetime(df["order_date"], format="mixed", errors="coerce")
    prices = parse_currency(df["unit_price"])
    mask = (
        dates.notna()
        & df["customer_id"].isin(customers["customer_id"])
        & df["product_id"].isin(products["product_id"])
        & df["quantity"].gt(0)
        & prices.notna()
        & df["discount"].between(0, 0.5)
    )
    valid = (
        df.loc[mask]
        .assign(order_date=dates[mask], unit_price=prices[mask])
        .reset_index(drop=True)
    )
    invalid = df.loc[~mask].reset_index(drop=True)
    return valid, invalid
```

</details>

## 4. Feature engineering, strings, and dates

Feature engineering is where a clean table becomes useful. The exercises here focus
on `assign`, conditional logic, binning, string extraction, date parts, lookup joins,
and composing small transformations with `.pipe`.

Keep every step pure: the input frame goes in, a new frame comes out.

### 4.1 Add revenue with assign

Write `add_revenue(df)` that adds a rounded `revenue` column:
`quantity * unit_price * (1 - discount)`.

Assume `unit_price` is already numeric. Leave the input unchanged.

**Hint:** `df.assign(name=expression)` returns a new frame; no assignment or `inplace` needed.

In [ ]:
def add_revenue(df):
    ...

**Self-check**

In [ ]:
fixture = pd.DataFrame({"quantity": [2, 1], "unit_price": [10.0, 5.5], "discount": [0.10, 0.0]})
out = add_revenue(fixture)
assert out["revenue"].tolist() == [18.0, 5.5]
assert "revenue" not in fixture.columns
base = pd.DataFrame({"quantity": [3, 2], "unit_price": [20.0, 15.0], "discount": [0.2, 0.0]})
assert add_revenue(base)["revenue"].tolist() == [48.0, 30.0]
print("4.1 passed")

<details>
<summary>Show solution</summary>

```python
def add_revenue(df):
    return df.assign(
        revenue=(df["quantity"] * df["unit_price"] * (1 - df["discount"])).round(2)
    )
```

</details>

### 4.2 Classify discounts with np.select

Write `classify_discount(series)` returning a Series named `discount_band` with:
- `unknown` when missing;
- `invalid` when below 0 or above 0.5;
- `none` when exactly 0;
- `low` when above 0 and at most 0.1;
- `medium` when above 0.1 and at most 0.2;
- `high` otherwise.

Preserve the input index.

**Hint:** `np.select` chooses the first matching condition, so order the conditions carefully.

In [ ]:
def classify_discount(series):
    ...

**Self-check**

In [ ]:
sample = pd.Series([np.nan, -0.1, 0.6, 0.0, 0.1, 0.2, 0.3])
out = classify_discount(sample)
assert out.tolist() == ["unknown", "invalid", "invalid", "none", "low", "medium", "high"]
assert out.index.equals(sample.index)
assert out.name == "discount_band"
actual = classify_discount(ORDERS["discount"])
assert actual.loc[ORDERS["discount"].isna()].eq("unknown").all()
assert actual.loc[ORDERS["discount"].lt(0) | ORDERS["discount"].gt(0.5)].eq("invalid").all()
print("4.2 passed")

<details>
<summary>Show solution</summary>

```python
def classify_discount(series):
    values = np.select(
        [
            series.isna(),
            series.lt(0) | series.gt(0.5),
            series.eq(0),
            series.le(0.1),
            series.le(0.2),
        ],
        ["unknown", "invalid", "none", "low", "medium"],
        default="high",
    )
    return pd.Series(values, index=series.index, name="discount_band")
```

</details>

### 4.3 Bin and quartile order value

Write `add_value_bands(df)` that uses the numeric `revenue` column to add:
- `value_band`: ordered bins `low` (up to 50), `medium` (up to 200), `high`
  (up to 500), and `premium` (above 500);
- `value_quartile`: four ranked quarters labelled `Q1` through `Q4`.

Missing revenue should stay missing. Leave the input unchanged.

**Hint:** Use `pd.cut` for explicit boundaries and `pd.qcut` for equal-sized groups.

In [ ]:
def add_value_bands(df):
    ...

**Self-check**

In [ ]:
fixture = pd.DataFrame({"revenue": [10, 100, 300, 1000, 50, 200, 500, 700, np.nan]})
out = add_value_bands(fixture)
assert out["value_band"].cat.ordered
assert out["value_band"].astype("string").tolist()[:8] == [
    "low", "medium", "high", "premium", "low", "medium", "high", "premium",
]
assert out["value_band"].isna().sum() == 1
assert set(out["value_quartile"].dropna().astype(str)) == {"Q1", "Q2", "Q3", "Q4"}
assert "value_band" not in fixture.columns
print("4.3 passed")

<details>
<summary>Show solution</summary>

```python
def add_value_bands(df):
    bands = pd.cut(
        df["revenue"],
        bins=[-np.inf, 50, 200, 500, np.inf],
        labels=["low", "medium", "high", "premium"],
    )
    quartiles = pd.qcut(
        df["revenue"].rank(method="first"),
        4,
        labels=["Q1", "Q2", "Q3", "Q4"],
    )
    return df.assign(value_band=bands, value_quartile=quartiles)
```

</details>

### 4.4 Extract and validate email parts

Write `extract_email_parts(series)` returning a DataFrame with columns
`email_norm`, `local_part`, `domain`, and `is_valid`.

Normalize by stripping and lowercasing. A simple valid email contains one `@`,
a non-empty local part, a non-empty domain, and a dot in the domain. For invalid
emails, set `local_part` and `domain` to missing.

**Hint:** `str.split("@", n=1, expand=True)` gives you local and domain parts in one step.

In [ ]:
def extract_email_parts(series):
    ...

**Self-check**

In [ ]:
sample = pd.Series([" Alice@Example.com ", "not-an-email", "bad@@example.com", None])
out = extract_email_parts(sample)
assert out["email_norm"].iloc[0] == "alice@example.com"
assert out["is_valid"].tolist() == [True, False, False, False]
assert out["domain"].iloc[0] == "example.com"
assert out["local_part"].iloc[1:].isna().all()
actual = extract_email_parts(CUSTOMERS["email"])
assert actual["is_valid"].sum() == 206
assert set(actual.loc[actual["is_valid"], "domain"].unique()) == {"example.com"}
assert actual.index.equals(CUSTOMERS.index)
print("4.4 passed")

<details>
<summary>Show solution</summary>

```python
def extract_email_parts(series):
    normalized = series.astype("string").str.strip().str.lower()
    valid = normalized.str.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", na=False)
    parts = normalized.str.split("@", n=1, expand=True)
    local_part = parts[0].where(valid)
    if 1 in parts.columns:
        domain = parts[1].where(valid)
    else:
        domain = pd.Series(pd.NA, index=series.index, dtype="string")
    return pd.DataFrame(
        {
            "email_norm": normalized,
            "local_part": local_part,
            "domain": domain,
            "is_valid": valid,
        }
    )
```

</details>

### 4.5 Add date features

Write `add_date_features(df)` for a frame whose `order_date` column is already
datetime. Add:
- `order_year` as `Int16`;
- `order_month` as `Int8`;
- `weekday` with the day name;
- `is_weekend` as a boolean.

Leave the input unchanged.

**Hint:** The `.dt` accessor works only on datetime-like Series.

In [ ]:
def add_date_features(df):
    ...

**Self-check**

In [ ]:
fixture = pd.DataFrame({"order_date": pd.to_datetime(["2024-01-05", "2024-01-06", "2024-01-07"])})
out = add_date_features(fixture)
assert out["order_year"].tolist() == [2024, 2024, 2024]
assert out["order_month"].tolist() == [1, 1, 1]
assert out["weekday"].tolist() == ["Friday", "Saturday", "Sunday"]
assert out["is_weekend"].tolist() == [False, True, True]
assert str(out["order_year"].dtype) == "Int16"
assert "weekday" not in fixture.columns
print("4.5 passed")

<details>
<summary>Show solution</summary>

```python
def add_date_features(df):
    dates = df["order_date"]
    return df.assign(
        order_year=dates.dt.year.astype("Int16"),
        order_month=dates.dt.month.astype("Int8"),
        weekday=dates.dt.day_name(),
        is_weekend=dates.dt.weekday.ge(5),
    )
```

</details>

### 4.6 Look up categories with map and merge

Implement two functions:
- `attach_category_map(orders, products)` using `Series.map`;
- `attach_category_merge(orders, products)` using `merge(..., validate="many_to_one")`.

Both should add a `category` column, keep every order row, and produce equivalent
results. Unknown or missing product IDs get a missing category.

**Hint:** `map` is a lightweight one-column lookup; `merge` validates cardinality and scales to many columns.

In [ ]:
def attach_category_map(orders, products):
    ...

def attach_category_merge(orders, products):
    ...

**Self-check**

In [ ]:
mapped = attach_category_map(ORDERS, PRODUCTS)
merged = attach_category_merge(ORDERS, PRODUCTS)
assert len(mapped) == len(merged) == len(ORDERS)
assert list(mapped.columns) == list(merged.columns)
pd.testing.assert_frame_equal(mapped, merged, check_dtype=False)
valid_products = ORDERS["product_id"].isin(PRODUCTS["product_id"])
assert mapped.loc[valid_products, "category"].notna().all()
assert mapped.loc[~valid_products, "category"].isna().all()
print("4.6 passed")

<details>
<summary>Show solution</summary>

```python
def attach_category_map(orders, products):
    lookup = products.set_index("product_id")["category"]
    return orders.assign(category=orders["product_id"].map(lookup))


def attach_category_merge(orders, products):
    return orders.merge(
        products[["product_id", "category"]],
        on="product_id",
        how="left",
        validate="many_to_one",
    )
```

</details>

### 4.7 Compose a small pipeline with pipe

Write `prepare_orders(df)` that passes the frame through three local steps:
1. parse `unit_price` with `parse_currency`;
2. add `revenue`;
3. add an `is_high_value` boolean for revenue of at least 20.

Use `.pipe` for the composition. The input frame must remain unchanged.

**Hint:** `df.pipe(f)` calls `f(df)`, so each step can be a small reusable function.

In [ ]:
def prepare_orders(df):
    ...

**Self-check**

In [ ]:
fixture = pd.DataFrame(
    {
        "order_id": ["a", "b"],
        "quantity": [2, 1],
        "unit_price": ["$10.00", "5.50"],
        "discount": [0.10, 0.0],
    }
)
out = prepare_orders(fixture)
assert out["unit_price"].tolist() == [10.0, 5.5]
assert out["revenue"].tolist() == [18.0, 5.5]
assert out["is_high_value"].tolist() == [False, False]
assert fixture["unit_price"].tolist() == ["$10.00", "5.50"]
assert "revenue" not in fixture.columns
print("4.7 passed")

<details>
<summary>Show solution</summary>

```python
def prepare_orders(df):
    def with_price(frame):
        return frame.assign(unit_price=parse_currency(frame["unit_price"]))

    def with_revenue(frame):
        return add_revenue(frame)

    def with_flag(frame):
        return frame.assign(is_high_value=frame["revenue"].ge(20))

    return df.pipe(with_price).pipe(with_revenue).pipe(with_flag)
```

</details>

## 5. Aggregation

`groupby` is the heart of pandas. This section moves from basic counts to named
aggregations, `transform` versus `agg`, group filtering, weighted metrics, and
multi-key summaries.

Prefer declarative aggregations over Python loops. When you need a per-group result
on the original row index, reach for `transform` rather than `apply`.

### 5.1 Summarize orders by status

Write `order_summary(orders)` returning one row per canonical status with:
- `orders`: number of rows;
- `customers`: unique customers;
- `units`: total quantity;
- `revenue`: total revenue.

Normalize the raw status labels first. Sort the index.

**Hint:** Named aggregation uses `output_name=("column", "function")` pairs.

In [ ]:
def order_summary(orders):
    ...

**Self-check**

In [ ]:
base = split_valid_orders(ORDERS, CUSTOMERS, PRODUCTS)[0].pipe(add_revenue)
out = order_summary(base)
assert list(out.columns) == ["orders", "customers", "units", "revenue"]
assert set(out.index) == {"completed", "cancelled", "pending", "returned"}
assert out["orders"].sum() == len(base)
assert out["customers"].sum() >= base["customer_id"].nunique()
np.testing.assert_allclose(out["revenue"].sum(), base["revenue"].sum())
assert out.index.is_monotonic_increasing
print("5.1 passed")

<details>
<summary>Show solution</summary>

```python
def order_summary(orders):
    status = (
        orders["status"].astype("string").str.strip().str.lower()
        .replace({"complete": "completed", "canceled": "cancelled"})
    )
    return (
        orders.assign(status=status)
        .groupby("status", observed=True)
        .agg(
            orders=("order_id", "size"),
            customers=("customer_id", "nunique"),
            units=("quantity", "sum"),
            revenue=("revenue", "sum"),
        )
        .sort_index()
    )
```

</details>

### 5.2 Build customer metrics

Write `customer_metrics(orders)` returning one row per customer with:
`n_orders`, `total_revenue`, `avg_order_value`, `first_order`, and `last_order`.

Count distinct order IDs, not raw rows. Sort by customer ID.

**Hint:** `nunique` is the right function when duplicates may exist.

In [ ]:
def customer_metrics(orders):
    ...

**Self-check**

In [ ]:
base = split_valid_orders(ORDERS, CUSTOMERS, PRODUCTS)[0].pipe(add_revenue)
out = customer_metrics(base)
assert list(out.columns) == ["n_orders", "total_revenue", "avg_order_value", "first_order", "last_order"]
assert out.index.is_unique
assert out["n_orders"].sum() == base["order_id"].nunique()
np.testing.assert_allclose(out["total_revenue"].sum(), base["revenue"].sum())
assert out["first_order"].le(out["last_order"]).all()
assert str(out["first_order"].dtype).startswith("datetime64")
print("5.2 passed")

<details>
<summary>Show solution</summary>

```python
def customer_metrics(orders):
    return (
        orders.groupby("customer_id")
        .agg(
            n_orders=("order_id", "nunique"),
            total_revenue=("revenue", "sum"),
            avg_order_value=("revenue", "mean"),
            first_order=("order_date", "min"),
            last_order=("order_date", "max"),
        )
        .sort_index()
    )
```

</details>

### 5.3 Add customer revenue share with transform

Write `add_customer_share(orders)` that adds:
- `customer_revenue`: total revenue for the order's customer;
- `revenue_share`: this row's revenue divided by that customer total.

The result must have the same number of rows and index as the input.

**Hint:** `transform("sum")` broadcasts the group total back to every row.

In [ ]:
def add_customer_share(orders):
    ...

**Self-check**

In [ ]:
base = split_valid_orders(ORDERS, CUSTOMERS, PRODUCTS)[0].pipe(add_revenue)
out = add_customer_share(base)
assert len(out) == len(base)
assert out.index.equals(base.index)
per_customer = out.groupby("customer_id")["revenue_share"].sum()
np.testing.assert_allclose(per_customer, 1.0)
assert out["customer_revenue"].groupby(out["customer_id"]).nunique().eq(1).all()
assert "revenue_share" not in base.columns
print("5.3 passed")

<details>
<summary>Show solution</summary>

```python
def add_customer_share(orders):
    customer_revenue = orders.groupby("customer_id")["revenue"].transform("sum")
    return orders.assign(
        customer_revenue=customer_revenue,
        revenue_share=orders["revenue"] / customer_revenue,
    )
```

</details>

### 5.4 Filter customers by activity

Write `filter_active_customers(orders, min_orders=3, min_revenue=1000.0)` that
returns every order row belonging to customers with at least `min_orders` distinct
orders and at least `min_revenue` total revenue.

Use a grouped filter rather than building a separate list of IDs.

**Hint:** `groupby(...).filter(func)` keeps or drops whole groups based on a boolean.

In [ ]:
def filter_active_customers(orders, min_orders=3, min_revenue=1000.0):
    ...

**Self-check**

In [ ]:
base = split_valid_orders(ORDERS, CUSTOMERS, PRODUCTS)[0].pipe(add_revenue)
out = filter_active_customers(base)
stats = base.groupby("customer_id").agg(
    n_orders=("order_id", "nunique"),
    revenue=("revenue", "sum"),
)
keep = stats.index[(stats["n_orders"] >= 3) & (stats["revenue"] >= 1000)]
assert set(out["customer_id"]) == set(keep)
assert len(out) == len(base[base["customer_id"].isin(keep)])
assert out["customer_id"].nunique() == len(keep)
print("5.4 passed")

<details>
<summary>Show solution</summary>

```python
def filter_active_customers(orders, min_orders=3, min_revenue=1000.0):
    return orders.groupby("customer_id").filter(
        lambda group: (
            group["order_id"].nunique() >= min_orders
            and group["revenue"].sum() >= min_revenue
        )
    )
```

</details>

### 5.5 Weighted average price by category

Write `weighted_average_price(orders)` returning a Series named
`weighted_avg_price` indexed by product category. Weight numeric `unit_price` by
`quantity` and ignore rows missing category, price, or quantity.

Compute the numerator and denominator with vectorized group operations, not a
per-group Python loop.

**Hint:** A weighted average is `sum(value * weight) / sum(weight)`.

In [ ]:
def weighted_average_price(orders):
    ...

**Self-check**

In [ ]:
category_map = clean_products(PRODUCTS)[["product_id", "category"]]
base = (
    split_valid_orders(ORDERS, CUSTOMERS, PRODUCTS)[0]
    .pipe(add_revenue)
    .merge(category_map, on="product_id", how="left", validate="many_to_one")
)
out = weighted_average_price(base)
assert out.name == "weighted_avg_price"
assert set(out.index) == {"Electronics", "Home Office", "Accessories", "Furniture"}
for category, group in base.dropna(subset=["category", "unit_price", "quantity"]).groupby("category"):
    expected = np.average(group["unit_price"], weights=group["quantity"])
    assert np.isclose(out.loc[category], expected)
print("5.5 passed")

<details>
<summary>Show solution</summary>

```python
def weighted_average_price(orders):
    data = orders.dropna(subset=["category", "unit_price", "quantity"])
    numerator = (data["unit_price"] * data["quantity"]).groupby(data["category"]).sum()
    denominator = data["quantity"].groupby(data["category"]).sum()
    return (numerator / denominator).rename("weighted_avg_price").sort_index()
```

</details>

### 5.6 Multi-key summary with within-channel shares

Write `multi_key_summary(orders)` that returns one row per canonical
`channel`/`status` combination with `orders`, `revenue`, and `channel_share`.

`channel_share` is the combination's revenue divided by the channel's total revenue.
Sort by channel and then descending revenue.

**Hint:** After `reset_index`, use another `groupby(...).transform` to normalize within channel.

In [ ]:
def multi_key_summary(orders):
    ...

**Self-check**

In [ ]:
base = split_valid_orders(ORDERS, CUSTOMERS, PRODUCTS)[0].pipe(add_revenue)
out = multi_key_summary(base)
assert list(out.columns) == ["channel", "status", "orders", "revenue", "channel_share"]
assert len(out) == 12
shares = out.groupby("channel")["channel_share"].sum()
np.testing.assert_allclose(shares, 1.0)
assert out["orders"].sum() == len(base)
assert out["revenue"].sum() == base["revenue"].sum()
print("5.6 passed")

<details>
<summary>Show solution</summary>

```python
def multi_key_summary(orders):
    data = orders.assign(
        status=(
            orders["status"].astype("string").str.strip().str.lower()
            .replace({"complete": "completed", "canceled": "cancelled"})
        ),
        channel=(
            orders["channel"].astype("string").str.strip().str.lower()
            .replace({"online": "web", "app": "mobile", "retail": "store"})
        ),
    )
    out = (
        data.groupby(["channel", "status"], observed=True)
        .agg(orders=("order_id", "size"), revenue=("revenue", "sum"))
        .reset_index()
    )
    out["channel_share"] = out.groupby("channel")["revenue"].transform(lambda values: values / values.sum())
    return out.sort_values(["channel", "revenue"], ascending=[True, False]).reset_index(drop=True)
```

</details>

### 5.7 Apply versus transform

Implement:
- `grouped_apply(orders)` computing a within-product z-score of `revenue` with
  `groupby(...).apply(...)`;
- `grouped_vectorized(orders)` computing the same thing with grouped `transform`.

Return both Series from `compare_grouped(orders)`. They should match.

**Hint:** With `apply`, the result may have a MultiIndex. With `transform`, it stays aligned to the input.

In [ ]:
def grouped_apply(orders):
    ...

def grouped_vectorized(orders):
    ...

def compare_grouped(orders):
    ...

**Self-check**

In [ ]:
base = split_valid_orders(ORDERS, CUSTOMERS, PRODUCTS)[0].pipe(add_revenue)
applied, vectorized = compare_grouped(base)
assert isinstance(applied, pd.Series) and isinstance(vectorized, pd.Series)
np.testing.assert_allclose(applied.sort_index(), vectorized.sort_index(), equal_nan=True)
assert len(applied) == len(base)
assert "zscore" not in base.columns
print("5.7 passed")

<details>
<summary>Show solution</summary>

```python
def grouped_apply(orders):
    return orders.groupby("product_id", group_keys=False)["revenue"].apply(
        lambda values: (values - values.mean()) / values.std(ddof=0)
    )


def grouped_vectorized(orders):
    grouped = orders.groupby("product_id")["revenue"]
    return (orders["revenue"] - grouped.transform("mean")) / grouped.transform("std", ddof=0)


def compare_grouped(orders):
    return grouped_apply(orders), grouped_vectorized(orders)
```

</details>

## 6. Combining and reshaping

Real datasets are relational. You will concatenate batches, join tables, audit join
cardinality, build anti-joins, and reshape between long and wide layouts.

`merge(validate=...)` is not decoration: it turns silent row multiplication into an
explicit error.

### 6.1 Concatenate batches with provenance

Write `concat_batches(frames)` where `frames` is a mapping from batch name to
DataFrame. Return one concatenated frame with a new `source` column identifying the
batch, a fresh RangeIndex, and the original frames untouched.

**Hint:** A dict comprehension with `assign(source=name)` keeps batch labels attached.

In [ ]:
def concat_batches(frames):
    ...

**Self-check**

In [ ]:
left = RETURNS.iloc[:60].copy()
right = RETURNS.iloc[60:100].copy()
out = concat_batches({"left": left, "right": right})
assert len(out) == 100
assert out["source"].value_counts().to_dict() == {"left": 60, "right": 40}
assert out.index.equals(pd.RangeIndex(100))
assert "source" not in left.columns and "source" not in right.columns
print("6.1 passed")

<details>
<summary>Show solution</summary>

```python
def concat_batches(frames):
    return pd.concat(
        [frame.assign(source=name) for name, frame in frames.items()],
        ignore_index=True,
    )
```

</details>

### 6.2 Left join orders to customers

Write `join_customers_orders(customers, orders)` that keeps every order, adds
customer columns, validates a many-to-one relationship, and includes an indicator
column. Suffix any overlapping names with `_customer`.

**Hint:** `indicator=True` adds `_merge` with `both`, `left_only`, or `right_only`.

In [ ]:
def join_customers_orders(customers, orders):
    ...

**Self-check**

In [ ]:
out = join_customers_orders(CUSTOMERS, ORDERS)
assert len(out) == len(ORDERS)
assert "_merge" in out.columns
assert out["_merge"].eq("both").sum() == 1683
assert out["_merge"].eq("left_only").sum() == 87
assert out.loc[out["_merge"] == "left_only", "customer_id"].isna().sum() + out.loc[out["_merge"] == "left_only", "customer_id"].eq("C9999").sum() == 87
print("6.2 passed")

<details>
<summary>Show solution</summary>

```python
def join_customers_orders(customers, orders):
    return orders.merge(
        customers,
        on="customer_id",
        how="left",
        validate="many_to_one",
        indicator=True,
        suffixes=("", "_customer"),
    )
```

</details>

### 6.3 Compare join types

Write `join_type_counts(customers, orders)` returning a dict with the row counts for
`inner`, `left`, `right`, and `outer` joins on `customer_id`, plus `anti` for
customers with no orders.

Use only the `customer_id` column from customers for the join counts.

**Hint:** A left anti-join is a left join followed by filtering on `_merge == "left_only"`.

In [ ]:
def join_type_counts(customers, orders):
    ...

**Self-check**

In [ ]:
out = join_type_counts(CUSTOMERS, ORDERS)
assert out == {"inner": 1683, "left": 1770, "right": 1698, "outer": 1785, "anti": 15}
print("6.3 passed")

<details>
<summary>Show solution</summary>

```python
def join_type_counts(customers, orders):
    keys = customers[["customer_id"]]
    counts = {}
    for how in ["inner", "left", "right", "outer"]:
        counts[how] = len(orders.merge(keys, on="customer_id", how=how))
    anti = customers.merge(
        orders[["customer_id"]].dropna().drop_duplicates(),
        on="customer_id",
        how="left",
        indicator=True,
    )
    counts["anti"] = int(anti["_merge"].eq("left_only").sum())
    return counts
```

</details>

### 6.4 Find orders with unknown products

Write `find_orphan_orders(orders, products)` returning order rows whose `product_id`
does not exist in products, including rows with a missing product ID.

Use an indicator merge and drop the indicator column from the result.

**Hint:** Missing keys also fail `isin`, so the indicator correctly marks them as left-only.

In [ ]:
def find_orphan_orders(orders, products):
    ...

**Self-check**

In [ ]:
out = find_orphan_orders(ORDERS, PRODUCTS)
assert len(out) == 35
assert "_merge" not in out.columns
assert not out["product_id"].isin(PRODUCTS["product_id"]).any()
assert out["product_id"].isna().sum() == 18
print("6.4 passed")

<details>
<summary>Show solution</summary>

```python
def find_orphan_orders(orders, products):
    keys = products[["product_id"]].drop_duplicates()
    merged = orders.merge(keys, on="product_id", how="left", indicator=True)
    return merged.loc[merged["_merge"].eq("left_only")].drop(columns="_merge")
```

</details>

### 6.5 Pivot monthly revenue by channel

Write `pivot_monthly_revenue(orders)` returning a wide table with month-end periods
as rows, canonical channels as columns, summed revenue as values, zeros for missing
combinations, and `Total` margins.

**Hint:** `pivot_table` supports `margins=True`; `pivot` does not aggregate and has no margins.

In [ ]:
def pivot_monthly_revenue(orders):
    ...

**Self-check**

In [ ]:
base = split_valid_orders(ORDERS, CUSTOMERS, PRODUCTS)[0].pipe(add_revenue)
out = pivot_monthly_revenue(base)
assert out.index.name == "month"
assert out.columns.name == "channel"
assert len(out) == 37
assert set(out.columns) == {"mobile", "store", "web", "Total"}
assert out.loc["Total", "Total"] == base["revenue"].sum()
assert out.drop(index="Total", columns="Total").notna().all().all()
print("6.5 passed")

<details>
<summary>Show solution</summary>

```python
def pivot_monthly_revenue(orders):
    data = orders.assign(
        month=orders["order_date"].dt.to_period("M"),
        channel=(
            orders["channel"].astype("string").str.strip().str.lower()
            .replace({"online": "web", "app": "mobile", "retail": "store"})
        ),
    )
    return data.pivot_table(
        index="month",
        columns="channel",
        values="revenue",
        aggfunc="sum",
        fill_value=0,
        margins=True,
        margins_name="Total",
    )
```

</details>

### 6.6 Melt a wide report back to long

Write `melt_report(wide)` that turns a wide frame with a `month` column and channel
columns into long format with columns `month`, `channel`, and `revenue`.

Drop rows whose revenue is missing. Keep the original wide frame unchanged.

**Hint:** `melt` is the inverse of a pivot: identifier columns stay, value columns become rows.

In [ ]:
def melt_report(wide):
    ...

**Self-check**

In [ ]:
base = split_valid_orders(ORDERS, CUSTOMERS, PRODUCTS)[0].pipe(add_revenue)
wide = pivot_monthly_revenue(base).drop(index="Total").drop(columns="Total")
out = melt_report(wide)
assert list(out.columns) == ["month", "channel", "revenue"]
assert len(out) == 108
assert out["revenue"].notna().all()
assert np.isclose(out["revenue"].sum(), wide.to_numpy().sum())
assert "month" in wide.index.names or wide.index.name == "month"
print("6.6 passed")

<details>
<summary>Show solution</summary>

```python
def melt_report(wide):
    return (
        wide.reset_index()
        .melt(id_vars="month", var_name="channel", value_name="revenue")
        .dropna(subset=["revenue"])
    )
```

</details>

### 6.7 Cross-tabulate status by channel

Write `crosstab_status_channel(orders)` returning row-normalized status proportions
for each canonical channel, with `All` margins.

Round to three decimals.

**Hint:** `normalize="index"` makes each row sum to one; margins add the overall distribution.

In [ ]:
def crosstab_status_channel(orders):
    ...

**Self-check**

In [ ]:
base = split_valid_orders(ORDERS, CUSTOMERS, PRODUCTS)[0]
out = crosstab_status_channel(base)
assert out.index.name == "channel"
assert out.columns.name == "status"
assert set(out.index) == {"mobile", "store", "web", "All"}
assert {"completed", "cancelled", "pending", "returned"}.issubset(out.columns)
row_sums = out.drop(index="All").sum(axis=1)
np.testing.assert_allclose(row_sums, 1.0, atol=0.002)
assert np.isclose(out.loc["All"].sum(), 1.0, atol=0.002)
print("6.7 passed")

<details>
<summary>Show solution</summary>

```python
def crosstab_status_channel(orders):
    data = orders.assign(
        status=(
            orders["status"].astype("string").str.strip().str.lower()
            .replace({"complete": "completed", "canceled": "cancelled"})
        ),
        channel=(
            orders["channel"].astype("string").str.strip().str.lower()
            .replace({"online": "web", "app": "mobile", "retail": "store"})
        ),
    )
    return pd.crosstab(
        data["channel"],
        data["status"],
        normalize="index",
        margins=True,
        margins_name="All",
    ).round(3)
```

</details>

## 7. Time series

Dates unlock aggregation over time. Here you build complete daily calendars,
resample, compute rolling and cumulative metrics, measure growth, find business-day
gaps, and group by quarters.

The key distinction is between observed dates and a complete calendar. Most time
series bugs come from silently skipping empty days or months.

### 7.1 Build daily and monthly revenue

Write `daily_monthly_revenue(orders)` returning `(daily, full_daily, monthly)`:
- `daily`: observed daily revenue;
- `full_daily`: the daily series reindexed to every calendar day in the observed
  range, with missing days filled with 0;
- `monthly`: month-end totals from `full_daily`.

Use `ME` as the month-end frequency alias.

**Hint:** `reindex` creates the missing calendar rows; `resample` then aggregates complete periods.

In [ ]:
def daily_monthly_revenue(orders):
    ...

**Self-check**

In [ ]:
base = split_valid_orders(ORDERS, CUSTOMERS, PRODUCTS)[0].pipe(add_revenue)
daily, full_daily, monthly = daily_monthly_revenue(base)
assert len(daily) == 788
assert len(full_daily) == 1096
assert str(full_daily.index.freq) == "<Day>"
assert len(monthly) == 36
assert str(monthly.index.freq) == "<MonthEnd>"
assert daily.sum() == full_daily.sum() == monthly.sum()
assert full_daily.notna().all()
assert monthly.index[0] == pd.Timestamp("2022-01-31")
assert monthly.index[-1] == pd.Timestamp("2024-12-31")
print("7.1 passed")

<details>
<summary>Show solution</summary>

```python
def daily_monthly_revenue(orders):
    daily = (
        orders.groupby(orders["order_date"].dt.normalize())["revenue"]
        .sum()
        .sort_index()
    )
    calendar = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
    full_daily = daily.reindex(calendar, fill_value=0)
    monthly = full_daily.resample("ME").sum()
    return daily, full_daily, monthly
```

</details>

### 7.2 Rolling and cumulative revenue

Write `rolling_metrics(daily, window=7)` returning a DataFrame with:
- `revenue`;
- `rolling_mean`: a centered-on-the-past rolling mean with `min_periods=1`;
- `cumulative`: cumulative revenue.

The index must match the input Series.

**Hint:** `min_periods=1` avoids a long leading block of missing rolling values.

In [ ]:
def rolling_metrics(daily, window=7):
    ...

**Self-check**

In [ ]:
base = split_valid_orders(ORDERS, CUSTOMERS, PRODUCTS)[0].pipe(add_revenue)
daily, full_daily, _ = daily_monthly_revenue(base)
out = rolling_metrics(full_daily)
assert list(out.columns) == ["revenue", "rolling_mean", "cumulative"]
assert out.index.equals(full_daily.index)
assert out["rolling_mean"].notna().all()
assert out["rolling_mean"].iloc[0] == full_daily.iloc[0]
assert np.isclose(out["rolling_mean"].iloc[-1], full_daily.iloc[-7:].mean())
assert out["cumulative"].iloc[-1] == full_daily.sum()
assert "rolling_mean" not in full_daily.index
print("7.2 passed")

<details>
<summary>Show solution</summary>

```python
def rolling_metrics(daily, window=7):
    return pd.DataFrame(
        {
            "revenue": daily,
            "rolling_mean": daily.rolling(window, min_periods=1).mean(),
            "cumulative": daily.cumsum(),
        }
    )
```

</details>

### 7.3 Monthly growth metrics

Write `monthly_growth(monthly)` returning a DataFrame with `revenue`,
`abs_change`, and `pct_change`.

Use the Series' own order. The first change values should be missing.

**Hint:** `diff()` measures absolute change; `pct_change()` measures relative change.

In [ ]:
def monthly_growth(monthly):
    ...

**Self-check**

In [ ]:
base = split_valid_orders(ORDERS, CUSTOMERS, PRODUCTS)[0].pipe(add_revenue)
monthly = daily_monthly_revenue(base)[2]
out = monthly_growth(monthly)
assert list(out.columns) == ["revenue", "abs_change", "pct_change"]
assert pd.isna(out["abs_change"].iloc[0]) and pd.isna(out["pct_change"].iloc[0])
assert out["abs_change"].iloc[1] == monthly.iloc[1] - monthly.iloc[0]
assert np.isclose(out["pct_change"].iloc[1], monthly.iloc[1] / monthly.iloc[0] - 1)
assert len(out) == len(monthly)
print("7.3 passed")

<details>
<summary>Show solution</summary>

```python
def monthly_growth(monthly):
    return pd.DataFrame(
        {
            "revenue": monthly,
            "abs_change": monthly.diff(),
            "pct_change": monthly.pct_change(),
        }
    )
```

</details>

### 7.4 Find missing business days

Write `business_day_gaps(daily)` returning `(business_daily, missing_days)`:
- `business_daily`: the observed daily Series reindexed to business days;
- `missing_days`: the business-day index positions with no observed data.

**Hint:** A business-day calendar uses the `"B"` frequency alias.

In [ ]:
def business_day_gaps(daily):
    ...

**Self-check**

In [ ]:
base = split_valid_orders(ORDERS, CUSTOMERS, PRODUCTS)[0].pipe(add_revenue)
daily = daily_monthly_revenue(base)[0]
business_daily, missing_days = business_day_gaps(daily)
assert len(business_daily) == 782
assert business_daily.index.dayofweek.max() < 5
expected_missing = pd.date_range(daily.index.min(), daily.index.max(), freq="B").difference(daily.index)
assert missing_days.equals(expected_missing)
assert len(missing_days) > 0
assert business_daily.loc[missing_days].isna().all()
print("7.4 passed")

<details>
<summary>Show solution</summary>

```python
def business_day_gaps(daily):
    business_days = pd.date_range(daily.index.min(), daily.index.max(), freq="B")
    business_daily = daily.reindex(business_days)
    return business_daily, business_daily[business_daily.isna()].index
```

</details>

### 7.5 Quarterly revenue by channel

Write `quarterly_channel_revenue(orders)` returning a wide table with quarter
periods as rows, canonical channels as columns, and summed revenue as values.
Fill absent combinations with 0.

**Hint:** `dt.to_period("Q")` gives calendar-quarter labels that sort correctly.

In [ ]:
def quarterly_channel_revenue(orders):
    ...

**Self-check**

In [ ]:
base = split_valid_orders(ORDERS, CUSTOMERS, PRODUCTS)[0].pipe(add_revenue)
out = quarterly_channel_revenue(base)
assert len(out) == 12
assert out.index.name == "quarter"
assert out.index.dtype == "period[Q-DEC]"
assert set(out.columns) == {"web", "mobile", "store"}
assert np.isclose(out.to_numpy().sum(), base["revenue"].sum())
print("7.5 passed")

<details>
<summary>Show solution</summary>

```python
def quarterly_channel_revenue(orders):
    data = orders.assign(
        quarter=orders["order_date"].dt.to_period("Q"),
        channel=(
            orders["channel"].astype("string").str.strip().str.lower()
            .replace({"online": "web", "app": "mobile", "retail": "store"})
        ),
    )
    return data.pivot_table(
        index="quarter",
        columns="channel",
        values="revenue",
        aggfunc="sum",
        fill_value=0,
    )
```

</details>

## 8. Advanced indexing

MultiIndexes make it possible to represent grouped results without losing structure.
This short section practices building them, selecting with `xs` and `.loc`, reshaping
with `unstack`, and enforcing a meaningful category order.

### 8.1 Slice a MultiIndex

Write `revenue_by_country_category(orders)` returning `(revenue, us, electronics)`:
- `revenue`: total revenue indexed by `country` and `category`, sorted;
- `us`: the country slice for `United States`;
- `electronics`: the category slice for `Electronics` via `xs`.

Drop rows missing country or category first.

**Hint:** `series.loc["United States"]` keeps the inner index; `xs` works on either level.

In [ ]:
def revenue_by_country_category(orders):
    ...

**Self-check**

In [ ]:
country = clean_country(CUSTOMERS["country"]).fillna("Unknown")
customers_clean = CUSTOMERS.assign(country=country)
category = clean_products(PRODUCTS)[["product_id", "category"]]
base = (
    split_valid_orders(ORDERS, CUSTOMERS, PRODUCTS)[0]
    .pipe(add_revenue)
    .merge(customers_clean[["customer_id", "country"]], on="customer_id", how="left", validate="many_to_one")
    .merge(category, on="product_id", how="left", validate="many_to_one")
)
revenue, us, electronics = revenue_by_country_category(base)
assert revenue.index.names == ["country", "category"]
assert isinstance(us, pd.Series) and isinstance(electronics, pd.Series)
assert len(us) == 4
assert len(electronics) == 6
assert us.index.name == "category"
assert electronics.index.name == "country"
assert np.isclose(revenue.sum(), base.dropna(subset=["country", "category"])["revenue"].sum())
print("8.1 passed")

<details>
<summary>Show solution</summary>

```python
def revenue_by_country_category(orders):
    data = orders.dropna(subset=["country", "category"])
    revenue = data.groupby(["country", "category"])["revenue"].sum().sort_index()
    us = revenue.loc["United States"]
    electronics = revenue.xs("Electronics", level="category")
    return revenue, us, electronics
```

</details>

### 8.2 Unstack a MultiIndex Series

Write `reshape_multiindex(series)` that unstacks the `category` level, names the
axis labels `country` and `category`, and sorts the result.

**Hint:** `unstack` converts the innermost index level into columns; `stack` reverses it.

In [ ]:
def reshape_multiindex(series):
    ...

**Self-check**

In [ ]:
country = clean_country(CUSTOMERS["country"]).fillna("Unknown")
customers_clean = CUSTOMERS.assign(country=country)
category = clean_products(PRODUCTS)[["product_id", "category"]]
base = (
    split_valid_orders(ORDERS, CUSTOMERS, PRODUCTS)[0]
    .pipe(add_revenue)
    .merge(customers_clean[["customer_id", "country"]], on="customer_id", how="left", validate="many_to_one")
    .merge(category, on="product_id", how="left", validate="many_to_one")
)
series = revenue_by_country_category(base)[0]
out = reshape_multiindex(series)
assert isinstance(out, pd.DataFrame)
assert out.index.name == "country"
assert out.columns.name == "category"
assert set(out.columns) == {"Accessories", "Electronics", "Furniture", "Home Office"}
assert out.index.is_monotonic_increasing
assert np.isclose(out.to_numpy().sum(), series.sum())
pd.testing.assert_series_equal(out.stack(), series, check_names=False)
print("8.2 passed")

<details>
<summary>Show solution</summary>

```python
def reshape_multiindex(series):
    return (
        series.unstack("category")
        .rename_axis(index="country", columns="category")
        .sort_index()
    )
```

</details>

### 8.3 Sort with an ordered category

Write `ordered_segments(customers)` that canonicalizes segment labels and converts
them to an ordered categorical with `Consumer < SMB < Enterprise`.

Sort by the category so the business order is respected, with missing segments last.

**Hint:** `pd.Categorical(..., ordered=True)` carries the business order into sorting.

In [ ]:
def ordered_segments(customers):
    ...

**Self-check**

In [ ]:
out = ordered_segments(CUSTOMERS)
assert isinstance(out["segment"].dtype, pd.CategoricalDtype)
assert out["segment"].dtype.ordered
assert list(out["segment"].cat.categories) == ["Consumer", "SMB", "Enterprise"]
assert out["segment"].value_counts(dropna=True).to_dict() == {
    "Consumer": 59,
    "SMB": 73,
    "Enterprise": 79,
}
assert out["segment"].isna().sum() == 4
assert out["segment"].dropna().is_monotonic_increasing
assert out.loc[out["segment"].isna()].index.min() > out.loc[out["segment"].notna()].index.max()
print("8.3 passed")

<details>
<summary>Show solution</summary>

```python
def ordered_segments(customers):
    mapping = {
        "consumer": "Consumer",
        "smb": "SMB",
        "small business": "SMB",
        "enterprise": "Enterprise",
        "ent": "Enterprise",
    }
    segment = (
        customers["segment"].astype("string").str.strip().str.lower()
        .map(mapping).astype("string")
    )
    ordered = pd.Categorical(
        segment,
        categories=["Consumer", "SMB", "Enterprise"],
        ordered=True,
    )
    return (
        customers.assign(segment=ordered)
        .sort_values("segment")
        .reset_index(drop=True)
    )
```

</details>

## 9. Performance and review

Idiomatic pandas is usually both clearer and faster. This section compares row-wise
Python with vectorized expressions, reduces numeric memory, and refactors an
imperative mutation-heavy function into a pure pipeline.

Timings vary by machine; correctness and readability are the primary goals.

### 9.1 Compare three revenue implementations

Write `compare_revenue_implementations(df)` returning `(iterrows_result,
apply_result, vectorized_result)` for `quantity * unit_price * (1 - discount)`.

Assume numeric columns and preserve the input index. Do not mutate the input.

**Hint:** The vectorized expression is the one to keep; the other two are for comparison only.

In [ ]:
def compare_revenue_implementations(df):
    ...

**Self-check**

In [ ]:
base = split_valid_orders(ORDERS, CUSTOMERS, PRODUCTS)[0]
iterrows_result, apply_result, vectorized_result = compare_revenue_implementations(base)
assert len(vectorized_result) == len(base)
assert vectorized_result.index.equals(base.index)
np.testing.assert_allclose(iterrows_result, vectorized_result)
np.testing.assert_allclose(apply_result, vectorized_result)
assert "revenue" not in base.columns
print("9.1 passed")

<details>
<summary>Show solution</summary>

```python
def compare_revenue_implementations(df):
    iterrows_result = pd.Series(
        [
            row["quantity"] * row["unit_price"] * (1 - row["discount"])
            for _, row in df.iterrows()
        ],
        index=df.index,
    )
    apply_result = df.apply(
        lambda row: row["quantity"] * row["unit_price"] * (1 - row["discount"]),
        axis=1,
    )
    vectorized_result = df["quantity"] * df["unit_price"] * (1 - df["discount"])
    return iterrows_result, apply_result, vectorized_result
```

</details>

### 9.2 Downcast numeric columns

Write `downcast_numeric(df)` returning a copy where integer-like columns use the
smallest integer dtype and float columns use `Float32`. Preserve values and leave
the input unchanged.

**Hint:** `select_dtypes(include="number")` and `pd.to_numeric(..., downcast=...)` do the work.

In [ ]:
def downcast_numeric(df):
    ...

**Self-check**

In [ ]:
base = split_valid_orders(ORDERS, CUSTOMERS, PRODUCTS)[0]
before_memory = base.memory_usage(deep=True).sum()
out = downcast_numeric(base)
assert out.memory_usage(deep=True).sum() < before_memory
assert "quantity" not in base.select_dtypes(include="float32").columns
np.testing.assert_allclose(out["quantity"].astype("float64"), base["quantity"].astype("float64"))
np.testing.assert_allclose(out["unit_price"].astype("float64"), base["unit_price"].astype("float64"))
assert out.index.equals(base.index)
print("9.2 passed")

<details>
<summary>Show solution</summary>

```python
def downcast_numeric(df):
    out = df.copy()
    for column in out.select_dtypes(include="number").columns:
        if pd.api.types.is_integer_dtype(out[column]):
            out[column] = pd.to_numeric(out[column], downcast="integer")
        else:
            out[column] = pd.to_numeric(out[column], downcast="float")
    return out
```

</details>

### 9.3 Refactor an imperative transformation

The imperative version below mutates its input and mixes several steps:

```python
def add_revenue_flags(df):
    df["unit_price"] = parse_currency(df["unit_price"])
    df["revenue"] = df["quantity"] * df["unit_price"] * (1 - df["discount"])
    df["is_high_value"] = df["revenue"] >= 500
    return df
```

Rewrite it as a pure function with the same output columns. The input must remain
unchanged, and `is_high_value` should be a nullable boolean.

**Hint:** `assign` can add several columns at once, all derived from the immutable input.

In [ ]:
def add_revenue_flags(df):
    ...

**Self-check**

In [ ]:
before = ORDERS.copy(deep=True)
out = add_revenue_flags(ORDERS)
assert "revenue" in out.columns
assert "is_high_value" in out.columns
assert str(out["unit_price"].dtype) == "Float64"
assert str(out["is_high_value"].dtype) == "boolean"
assert out.loc[out["revenue"].isna(), "is_high_value"].isna().all()
assert out.loc[out["revenue"].ge(500), "is_high_value"].all()
assert "revenue" not in ORDERS.columns
pd.testing.assert_frame_equal(ORDERS, before)
print("9.3 passed")

<details>
<summary>Show solution</summary>

```python
def add_revenue_flags(df):
    prices = parse_currency(df["unit_price"])
    revenue = (df["quantity"] * prices * (1 - df["discount"])).round(2)
    return df.assign(
        unit_price=prices,
        revenue=revenue,
        is_high_value=revenue.ge(500),
    )
```

</details>

## 10. Functional capstone

This capstone turns the raw CSV files into a customer analytics product. Build the
pipeline in small pure steps, compose them with `.pipe`, and reconcile totals at the
end.

The five checkpoints are cumulative. Use the helpers you wrote earlier; if one is
missing, define a local replacement in the capstone cell.

### 10.1 Load all tables with explicit types

Write `load_tables(base_dir=DATA_DIR)` returning a dict with keys `customers`,
`products`, `prices`, `orders`, and `returns`.

Use `"string"` dtype for every ID column. Do not parse dates yet.

**Hint:** A dict of DataFrames is a convenient state object for a pipeline.

In [ ]:
def load_tables(base_dir=DATA_DIR):
    ...

**Self-check**

In [ ]:
tables = load_tables()
assert set(tables) == {"customers", "products", "prices", "orders", "returns"}
assert tables["customers"].shape == (215, 7)
assert tables["products"].shape == (30, 6)
assert tables["prices"].shape == (62, 3)
assert tables["orders"].shape == (1770, 10)
assert tables["returns"].shape == (170, 5)
assert str(tables["orders"]["order_id"].dtype) == "string"
assert str(tables["returns"]["order_id"].dtype) == "string"
print("10.1 passed")

<details>
<summary>Show solution</summary>

```python
def load_tables(base_dir=DATA_DIR):
    base = Path(base_dir)
    return {
        "customers": pd.read_csv(
            base / "customers.csv",
            dtype={"customer_id": "string"},
        ),
        "products": pd.read_csv(
            base / "products.csv",
            dtype={"product_id": "string"},
        ),
        "prices": pd.read_csv(
            base / "product_prices.csv",
            dtype={"product_id": "string"},
        ),
        "orders": pd.read_csv(
            base / "orders.csv",
            dtype={
                "order_id": "string",
                "customer_id": "string",
                "product_id": "string",
            },
        ),
        "returns": pd.read_csv(
            base / "returns.csv",
            dtype={"return_id": "string", "order_id": "string"},
        ),
    }
```

</details>

### 10.2 Clean every table without mutating raw data

Write `clean_tables(tables)` returning a dict with the same keys, where:
- customers have trimmed names, normalized emails, canonical countries and segments,
  parsed signup dates, nullable boolean opt-in, and unknown labels filled with
  `"Unknown"`;
- products come from `clean_products`;
- prices have parsed dates and sorted rows;
- orders are deduplicated, with parsed dates, parsed prices, canonical status and
  channel;
- returns have parsed dates, canonical reasons, and one row per `return_id`.

Do not mutate the input dict or its frames.

**Hint:** A helper that returns a fresh dict of fresh frames is easy to test for purity.

In [ ]:
def clean_tables(tables):
    ...

**Self-check**

In [ ]:
raw = load_tables()
before = {name: frame.copy(deep=True) for name, frame in raw.items()}
clean = clean_tables(raw)
assert set(clean) == set(raw)
assert clean["customers"].shape[0] == 215
assert clean["orders"].shape[0] == 1700
assert clean["returns"].shape[0] == 160
assert clean["orders"]["order_id"].is_unique
assert clean["returns"]["return_id"].is_unique
assert clean["customers"]["country"].notna().all()
assert clean["customers"]["segment"].notna().all()
assert str(clean["customers"]["marketing_opt_in"].dtype) == "boolean"
assert str(clean["orders"]["unit_price"].dtype) == "Float64"
assert str(clean["orders"]["order_date"].dtype).startswith("datetime64")
assert str(clean["orders"]["updated_at"].dtype).startswith("datetime64")
assert str(clean["prices"]["effective_date"].dtype).startswith("datetime64")
assert str(clean["returns"]["return_date"].dtype).startswith("datetime64")
assert clean["orders"]["order_date"].isna().sum() == 51
for name, frame in raw.items():
    pd.testing.assert_frame_equal(frame, before[name])
print("10.2 passed")

<details>
<summary>Show solution</summary>

```python
def clean_tables(tables):
    country = clean_country(tables["customers"]["country"]).fillna("Unknown")
    segment_map = {
        "consumer": "Consumer",
        "smb": "SMB",
        "small business": "SMB",
        "enterprise": "Enterprise",
        "ent": "Enterprise",
    }
    segment = (
        tables["customers"]["segment"].astype("string").str.strip().str.lower()
        .map(segment_map).fillna("Unknown").astype("string")
    )
    opt_in_map = {
        "y": True,
        "yes": True,
        "1": True,
        "true": True,
        "n": False,
        "no": False,
        "0": False,
        "false": False,
    }
    opt_in = (
        tables["customers"]["marketing_opt_in"].astype("string").str.strip().str.lower()
        .map(opt_in_map).astype("boolean")
    )
    customers = tables["customers"].assign(
        full_name=tables["customers"]["full_name"].str.strip(),
        email=tables["customers"]["email"].astype("string").str.strip().str.lower(),
        country=country,
        signup_date=pd.to_datetime(tables["customers"]["signup_date"], format="mixed", errors="coerce"),
        segment=segment,
        marketing_opt_in=opt_in,
    )

    products = clean_products(tables["products"])

    prices = tables["prices"].assign(
        effective_date=pd.to_datetime(tables["prices"]["effective_date"], format="mixed", errors="coerce"),
        list_price=pd.to_numeric(tables["prices"]["list_price"], errors="coerce"),
    ).sort_values(["product_id", "effective_date"]).reset_index(drop=True)

    orders = dedupe_orders(tables["orders"])
    orders = orders.assign(
        order_date=pd.to_datetime(orders["order_date"], format="mixed", errors="coerce"),
        updated_at=pd.to_datetime(orders["updated_at"], format="mixed", errors="coerce"),
        unit_price=parse_currency(orders["unit_price"]),
        status=(
            orders["status"].astype("string").str.strip().str.lower()
            .replace({"complete": "completed", "canceled": "cancelled"})
        ),
        channel=(
            orders["channel"].astype("string").str.strip().str.lower()
            .replace({"online": "web", "app": "mobile", "retail": "store"})
        ),
    )

    reason_map = {
        "damaged": "damaged",
        "damaged item": "damaged",
        "wrong item": "wrong item",
        "wrong-item": "wrong item",
        "size": "size",
        "changed mind": "changed mind",
        "changed_mind": "changed mind",
    }
    returns = tables["returns"].assign(
        return_date=pd.to_datetime(tables["returns"]["return_date"], format="mixed", errors="coerce"),
        reason=(
            tables["returns"]["reason"].astype("string").str.strip().str.lower()
            .map(reason_map).astype("string")
        ),
    )
    returns = (
        returns.sort_values(["return_id", "return_date"], na_position="first")
        .drop_duplicates("return_id", keep="last")
        .reset_index(drop=True)
    )

    return {
        "customers": customers,
        "products": products,
        "prices": prices,
        "orders": orders,
        "returns": returns,
    }
```

</details>

### 10.3 Build the fact table

Write `build_fact(clean_tables)` that keeps valid orders and enriches them with
customer country/segment and product category/unit_cost.

Valid means: parsed date, known customer, known product, positive quantity, parsed
price, and discount between 0 and 0.5. Add `revenue`, `margin`, `month`, and
`quarter`. Use `validate="many_to_one"` on both joins.

**Hint:** Filter first, then merge; it keeps the validation checks cheap and the fact table clean.

In [ ]:
def build_fact(clean_tables):
    ...

**Self-check**

In [ ]:
clean = clean_tables(load_tables())
fact = build_fact(clean)
assert len(fact) == 1480
assert fact["customer_id"].nunique() == 200
assert fact["product_id"].nunique() == 30
assert fact["country"].notna().all()
assert fact["category"].notna().all()
assert fact["quantity"].gt(0).all()
assert fact["unit_price"].notna().all()
assert fact["discount"].between(0, 0.5).all()
assert round(float(fact["revenue"].sum()), 2) == 1299800.79
assert str(fact["month"].dtype) == "period[M]"
assert str(fact["quarter"].dtype) == "period[Q-DEC]"
print("10.3 passed")

<details>
<summary>Show solution</summary>

```python
def build_fact(clean_tables):
    orders = clean_tables["orders"]
    valid = orders.loc[
        orders["order_date"].notna()
        & orders["customer_id"].isin(clean_tables["customers"]["customer_id"])
        & orders["product_id"].isin(clean_tables["products"]["product_id"])
        & orders["quantity"].gt(0)
        & orders["unit_price"].notna()
        & orders["discount"].between(0, 0.5)
    ].copy()
    fact = (
        valid.merge(
            clean_tables["customers"][["customer_id", "country", "segment"]],
            on="customer_id",
            how="left",
            validate="many_to_one",
        )
        .merge(
            clean_tables["products"][["product_id", "category", "unit_cost"]],
            on="product_id",
            how="left",
            validate="many_to_one",
        )
        .assign(
            revenue=lambda frame: (
                frame["quantity"] * frame["unit_price"] * (1 - frame["discount"])
            ).round(2),
            margin=lambda frame: (
                frame["quantity"] * frame["unit_price"] * (1 - frame["discount"])
                - frame["quantity"] * frame["unit_cost"]
            ).round(2),
            month=lambda frame: frame["order_date"].dt.to_period("M"),
            quarter=lambda frame: frame["order_date"].dt.to_period("Q"),
        )
    )
    return fact.reset_index(drop=True)
```

</details>

### 10.4 Customer analytics

Write `customer_analytics(fact)` returning one row per customer with:
`country`, `segment`, `n_orders`, `units`, `revenue`, `margin`, `avg_order_value`,
`first_order`, `last_order`, `recency_days`, and `margin_pct`.

Sort by customer ID. Do not include customers with no valid orders.

**Hint:** Aggregate first, then compute derived columns from the aggregated frame.

In [ ]:
def customer_analytics(fact):
    ...

**Self-check**

In [ ]:
fact = build_fact(clean_tables(load_tables()))
out = customer_analytics(fact)
assert len(out) == 200
assert out.index.name == "customer_id"
assert out.index.is_unique
assert list(out.columns) == [
    "country", "segment", "n_orders", "units", "revenue", "margin",
    "avg_order_value", "first_order", "last_order", "recency_days", "margin_pct",
]
assert np.isclose(out["revenue"].sum(), fact["revenue"].sum())
assert out["n_orders"].gt(0).all()
assert out["recency_days"].ge(0).all()
assert out["first_order"].le(out["last_order"]).all()
assert out["margin_pct"].dropna().between(-1, 1).all()
print("10.4 passed")

<details>
<summary>Show solution</summary>

```python
def customer_analytics(fact):
    customers = (
        fact.groupby("customer_id")
        .agg(
            country=("country", "first"),
            segment=("segment", "first"),
            n_orders=("order_id", "nunique"),
            units=("quantity", "sum"),
            revenue=("revenue", "sum"),
            margin=("margin", "sum"),
            avg_order_value=("revenue", "mean"),
            first_order=("order_date", "min"),
            last_order=("order_date", "max"),
        )
        .sort_index()
    )
    customers["recency_days"] = (fact["order_date"].max() - customers["last_order"]).dt.days
    customers["margin_pct"] = customers["margin"] / customers["revenue"]
    return customers
```

</details>

### 10.5 Build the report and compose the pipeline

Write two functions:
- `build_report(fact)` returning a dict with `monthly` (month × country revenue),
  `top_customers` (top 3 customers per country by revenue), and `category_share`
  (revenue and share by category);
- `run_pipeline(base_dir=DATA_DIR)` returning a dict with keys `clean`, `fact`,
  `customers`, and `report`; use `.pipe` for the DataFrame stages.

**Hint:** Keep `load_tables`, `clean_tables`, `build_fact`, and the report as independent pure steps.

In [ ]:
def build_report(fact):
    ...

def run_pipeline(base_dir=DATA_DIR):
    ...

**Self-check**

In [ ]:
result = run_pipeline()
assert set(result) == {"clean", "fact", "customers", "report"}
assert len(result["fact"]) == 1480
assert len(result["customers"]) == 200
assert set(result["report"]) == {"monthly", "top_customers", "category_share"}
monthly = result["report"]["monthly"]
assert monthly.shape == (36, 6)
assert np.isclose(monthly.to_numpy().sum(), result["fact"]["revenue"].sum())
top = result["report"]["top_customers"]
assert top.groupby("country").size().le(3).all()
assert set(top["country"]) == set(result["fact"]["country"])
assert len(top) == 18
shares = result["report"]["category_share"]
assert np.isclose(shares["share"].sum(), 1.0)
assert np.isclose(shares["revenue"].sum(), result["fact"]["revenue"].sum())
print("10.5 passed")

<details>
<summary>Show solution</summary>

```python
def build_report(fact):
    monthly = fact.pivot_table(
        index="month",
        columns="country",
        values="revenue",
        aggfunc="sum",
        fill_value=0,
    )
    top_customers = (
        fact.groupby(["country", "customer_id"])["revenue"]
        .sum()
        .reset_index()
        .assign(
            rank=lambda frame: frame.groupby("country")["revenue"].rank(
                method="first", ascending=False
            )
        )
        .query("rank <= 3")
        .sort_values(["country", "rank"])
        .reset_index(drop=True)
    )
    category_share = (
        fact.groupby("category")["revenue"]
        .sum()
        .to_frame("revenue")
        .assign(share=lambda frame: frame["revenue"] / frame["revenue"].sum())
        .sort_values("revenue", ascending=False)
    )
    return {
        "monthly": monthly,
        "top_customers": top_customers,
        "category_share": category_share,
    }


def run_pipeline(base_dir=DATA_DIR):
    clean = clean_tables(load_tables(base_dir))
    fact = build_fact(clean)
    return {
        "clean": clean,
        "fact": fact,
        "customers": fact.pipe(customer_analytics),
        "report": fact.pipe(build_report),
    }
```

</details>

## 11. Challenge set

These problems combine several techniques. There is more than one correct shape for
some answers, but the self-checks enforce the stated contract.

Try to keep the implementations declarative and readable.

### 11.1 RFM segmentation

Write `rfm_segments(customers, fact)` returning one row per customer with
`recency`, `frequency`, `monetary`, `R`, `F`, `M`, `rfm_score`, `country`,
`customer_segment`, and `rfm_segment`.

Use quartile scores from 1 to 4, where recent, frequent, and high-spend customers
score 4. Build a simple segment rule from the scores.

**Hint:** Rank before `qcut` so tied values do not create duplicate bin edges.

In [ ]:
def rfm_segments(customers, fact):
    ...

**Self-check**

In [ ]:
tables = clean_tables(load_tables())
fact = build_fact(tables)
out = rfm_segments(tables["customers"], fact)
assert len(out) == 200
assert out.index.name == "customer_id"
assert list(out.columns) == [
    "recency", "frequency", "monetary", "R", "F", "M",
    "rfm_score", "rfm_segment", "country", "customer_segment",
]
assert out[["R", "F", "M"]].isin([1, 2, 3, 4]).all().all()
assert out["rfm_score"].between(111, 444).all()
assert set(out["rfm_segment"]).issubset(
    {"Champions", "Loyal", "At risk", "Needs attention", "Hibernating"}
)
assert out["country"].notna().all()
assert out["customer_segment"].notna().all()
assert out["recency"].ge(0).all()
print("11.1 passed")

<details>
<summary>Show solution</summary>

```python
def rfm_segments(customers, fact):
    snapshot = fact["order_date"].max() + pd.Timedelta(days=1)
    rfm = (
        fact.groupby("customer_id")
        .agg(
            recency=("order_date", lambda values: (snapshot - values.max()).days),
            frequency=("order_id", "nunique"),
            monetary=("revenue", "sum"),
        )
    )
    rfm["R"] = pd.qcut(rfm["recency"].rank(method="first"), 4, labels=[4, 3, 2, 1]).astype("int8")
    rfm["F"] = pd.qcut(rfm["frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype("int8")
    rfm["M"] = pd.qcut(rfm["monetary"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype("int8")
    rfm["rfm_score"] = rfm["R"].astype(int) * 100 + rfm["F"].astype(int) * 10 + rfm["M"].astype(int)
    rfm["rfm_segment"] = np.select(
        [
            rfm["rfm_score"].ge(444),
            rfm["rfm_score"].ge(433),
            rfm["rfm_score"].ge(322),
            rfm["rfm_score"].ge(211),
        ],
        ["Champions", "Loyal", "At risk", "Needs attention"],
        default="Hibernating",
    )
    profile = customers.set_index("customer_id")[["country", "segment"]].rename(
        columns={"segment": "customer_segment"}
    )
    return rfm.join(profile, how="left")
```

</details>

### 11.2 Cohort retention matrix

Write `cohort_retention(fact)` returning a matrix with cohort month as rows,
periods since first order as columns, and the share of each cohort active in that
period.

Period 0 must be 1.0 for every cohort.

**Hint:** Add the cohort label and period number to every order row before counting unique customers.

In [ ]:
def cohort_retention(fact):
    ...

**Self-check**

In [ ]:
fact = build_fact(clean_tables(load_tables()))
out = cohort_retention(fact)
assert out.index.name == "cohort"
assert out.columns.name == "period"
assert len(out) == 19
assert out.columns.min() == 0
assert np.allclose(out[0], 1.0)
assert out.to_numpy().max() <= 1.0
assert out.to_numpy().min() >= 0.0
assert out.notna().all().all()
print("11.2 passed")

<details>
<summary>Show solution</summary>

```python
def cohort_retention(fact):
    data = fact.assign(order_month=fact["order_date"].dt.to_period("M"))
    data["cohort"] = data.groupby("customer_id")["order_month"].transform("min")
    data["period"] = (
        (data["order_month"].dt.year - data["cohort"].dt.year) * 12
        + (data["order_month"].dt.month - data["cohort"].dt.month)
    )
    counts = (
        data.groupby(["cohort", "period"])["customer_id"]
        .nunique()
        .unstack(fill_value=0)
    )
    sizes = data.groupby("cohort")["customer_id"].nunique()
    return counts.div(sizes, axis=0)
```

</details>

### 11.3 Channel funnel

Write `funnel_by_channel(fact)` returning status counts by channel, a row total, and
`completion_rate` and `cancel_rate` columns. Include an `All` margin row and column.

**Hint:** `crosstab` margins give both the row totals and the grand total.

In [ ]:
def funnel_by_channel(fact):
    ...

**Self-check**

In [ ]:
fact = build_fact(clean_tables(load_tables()))
out = funnel_by_channel(fact)
assert set(out.index) == {"mobile", "store", "web", "All"}
assert {"completed", "cancelled", "pending", "returned", "All"}.issubset(out.columns)
assert out.loc[["mobile", "store", "web"], "All"].sum() == len(fact)
assert out["completion_rate"].between(0, 1).all()
assert out["cancel_rate"].between(0, 1).all()
assert np.isclose(out.loc["All", "All"], len(fact))
print("11.3 passed")

<details>
<summary>Show solution</summary>

```python
def funnel_by_channel(fact):
    out = pd.crosstab(
        fact["channel"],
        fact["status"],
        margins=True,
        margins_name="All",
    )
    out["completion_rate"] = out["completed"] / out["All"]
    out["cancel_rate"] = out["cancelled"] / out["All"]
    return out
```

</details>

### 11.4 As-of price attachment

Write `attach_price_asof(orders, prices)` that attaches the most recent
`list_price` known on or before each order date for the order's product.

Use `merge_asof` with `by="product_id"` and `direction="backward"`. Return the
matched frame sorted by date. Use only orders with a parsed date and a known product.

**Hint:** Both frames must be sorted by the as-of key; `by` partitions the match by product.

In [ ]:
def attach_price_asof(orders, prices):
    ...

**Self-check**

In [ ]:
tables = clean_tables(load_tables())
orders = tables["orders"]
prices = tables["prices"]
out = attach_price_asof(orders, prices)
eligible = orders.dropna(subset=["order_date"])
eligible = eligible[eligible["product_id"].isin(prices["product_id"])]
assert len(out) == len(eligible)
assert out["list_price"].notna().all()
assert out["order_date"].is_monotonic_increasing
assert (out["effective_date"] <= out["order_date"]).all()
assert "list_price" in out.columns
sample = out.iloc[0]
expected = (
    prices.loc[
        (prices["product_id"] == sample["product_id"])
        & (prices["effective_date"] <= sample["order_date"]),
        "list_price",
    ]
    .iloc[-1]
)
assert np.isclose(sample["list_price"], expected)
print("11.4 passed")

<details>
<summary>Show solution</summary>

```python
def attach_price_asof(orders, prices):
    left = (
        orders.dropna(subset=["order_date"])
        .loc[orders["product_id"].isin(prices["product_id"])]
        .sort_values(["order_date", "product_id"])
    )
    right = prices.sort_values(["effective_date", "product_id"])
    return pd.merge_asof(
        left,
        right,
        left_on="order_date",
        right_on="effective_date",
        by="product_id",
        direction="backward",
        suffixes=("", "_price"),
    )
```

</details>

### 11.5 Deduplicate customers by normalized email

Write `dedupe_customers(customers)` keeping one row per non-empty normalized email.
When an email appears more than once, keep the customer with the latest parsed
signup date. Customers with missing email are all kept.

Return a fresh frame with a reset index.

**Hint:** Sort by signup date first, then keep the last occurrence of each non-empty normalized email.

In [ ]:
def dedupe_customers(customers):
    ...

**Self-check**

In [ ]:
customers = clean_tables(load_tables())["customers"]
before = customers.copy(deep=True)
out = dedupe_customers(customers)
assert len(out) == 209
assert out["email"].dropna().is_unique
assert out["customer_id"].is_unique
assert customers["email"].notna().sum() == before["email"].notna().sum()
for email, group in customers.dropna(subset=["email"]).groupby("email"):
    if len(group) > 1:
        kept = out.loc[out["email"] == email]
        assert len(kept) == 1
        assert kept["signup_date"].iloc[0] == group["signup_date"].max()
print("11.5 passed")

<details>
<summary>Show solution</summary>

```python
def dedupe_customers(customers):
    data = customers.assign(
        _email=customers["email"].astype("string").str.strip().str.lower(),
        _signup=pd.to_datetime(customers["signup_date"], format="mixed", errors="coerce"),
    )
    data = data.sort_values("_signup", na_position="first")
    keep = (
        ~data["_email"].duplicated(keep="last")
        | data["_email"].isna()
        | data["_email"].eq("")
    )
    return data.loc[keep].drop(columns=["_email", "_signup"]).reset_index(drop=True)
```

</details>

## 12. Cheat sheet and self-review

### Functional pandas patterns

| Goal | Pattern |
| --- | --- |
| Add derived columns | `df.assign(revenue=...)` |
| Filter rows | `df.loc[mask]`, `df.query("...")` |
| Select labels/positions | `.loc[row, column]`, `.iloc[row, column]` |
| Compose transformations | `df.pipe(step_one).pipe(step_two)` |
| Replace labels | `Series.map(dictionary)` or `Series.replace(mapping)` |
| Conditional labels | `np.select(conditions, choices, default=...)` |
| Grouped broadcast | `df.groupby(key)[col].transform("median")` |
| Named aggregation | `df.groupby(key).agg(total=("col", "sum"))` |
| Keep whole groups | `df.groupby(key).filter(...)` |
| Validate joins | `merge(..., validate="many_to_one")` |
| Audit joins | `merge(..., indicator=True)` |
| Reshape wide/long | `pivot_table`, `melt`, `stack`, `unstack` |
| Time aggregation | `resample("ME")`, `rolling`, `expanding` |
| Strings | `Series.str.strip/lower/contains/extract/split` |
| Dates | `Series.dt.year/month/day_name/normalize/to_period` |
| Ordered categories | `pd.Categorical(..., categories=..., ordered=True)` |
| Testing | `pd.testing.assert_frame_equal`, `assert_series_equal` |

### Code-review checklist

- Does every function return a new object instead of mutating its argument?
- Is every join's cardinality stated with `validate=`?
- Are missing keys and missing values handled deliberately?
- Are dates parsed once, explicitly, instead of repeatedly inside expressions?
- Are labels normalized before grouping or joining?
- Is there any `iterrows`, row-wise `apply`, or chained indexing that can be removed?
- Are numeric columns stored in memory-efficient dtypes where appropriate?
- Do totals reconcile after filtering, joining, and reshaping?

### What to practise next

- Replace the synthetic data with a real CSV and repeat the audit.
- Add pytest tests around the pure helper functions.
- Profile the capstone with `%timeit` and compare dtypes.
- Try the same transformations with Polars or DuckDB.